In [ ]:
# Basic Imports

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from collections import Counter
import os

In [ ]:
# --- HYPERPARAMETERS ---
WINDOW_SIZE = 100  # 1 second of data at 100Hz
STRIDE = 20        # 200ms overlap
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 0.001
CSV_FILENAME = "gesture_dataset_raw.csv"
MODEL_SAVE_PATH = "tiny_gesture_net.pth"

In [ ]:
class IMUDataset(Dataset):
    def __init__(self, csv_file, window_size, stride):
        self.window_size = window_size
        self.stride = stride

        # 1. Load data
        df = pd.DataFrame(columns=['timestamp', 'ax', 'ay', 'az', 'gx', 'gy', 'gz', 'label'])
        if os.path.exists(csv_file):
            df = pd.read_csv(csv_file)
        else:
            raise FileNotFoundError(f"Ensure {csv_file} is in the same directory.")

        # 2. Extract features and labels
        features = df[['ax', 'ay', 'az', 'gx', 'gy', 'gz']].values
        labels = df['label'].values

        # 3. Z-Score Normalization
        # Extremely important for uncalibrated data to center the gravity and bias offsets
        self.scaler = StandardScaler()
        features = self.scaler.fit_transform(features)

        self.x_windows = []
        self.y_labels = []

        # Slide over the sequence
        for i in range(0, len(features) - window_size, stride):
            window = features[i : i + window_size]
            window_labels = labels[i : i + window_size]

            # The label for the window is the most common label inside it (Mode)
            majority_label = Counter(window_labels).most_common(1)[0][0]

            # Transpose to (Channels, Sequence_Length) for PyTorch 1D Convs
            self.x_windows.append(window.T)
            self.y_labels.append(majority_label)

        self.x_windows = torch.tensor(np.array(self.x_windows), dtype=torch.float32)
        self.y_labels = torch.tensor(np.array(self.y_labels), dtype=torch.long)

    def __len__(self):
        return len(self.x_windows)

    def __getitem__(self, idx):
        return self.x_windows[idx], self.y_labels[idx]

In [ ]:
class TinyGestureNet(nn.Module):
    def __init__(self, num_classes=3):
        super(TinyGestureNet, self).__init__()

        # Block 1: Increased channels to 32, added BatchNorm
        self.conv1 = nn.Conv1d(in_channels=6, out_channels=32, kernel_size=5, stride=2, padding=2)
        self.bn1 = nn.BatchNorm1d(32) # Will be folded into conv1 during TFLite conversion
        self.relu1 = nn.ReLU()

        # Block 2: Depthwise Separable Convolution (Wider)
        self.depthwise = nn.Conv1d(in_channels=32, out_channels=32, kernel_size=5, padding=2, groups=32)
        self.pointwise = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(64) # Will be folded into pointwise
        self.relu2 = nn.ReLU()

        self.pool = nn.MaxPool1d(kernel_size=2, stride=2)
        self.gap = nn.AdaptiveAvgPool1d(1)

        # Added dropout to prevent overfitting to specific phone orientations
        self.dropout = nn.Dropout(0.3)

        # Final classifier
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.depthwise(x)
        x = self.relu2(self.bn2(self.pointwise(x)))
        x = self.pool(x)
        x = self.gap(x)
        x = x.view(x.size(0), -1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

In [ ]:
def train_model(epochs=EPOCHS):
    dataset = IMUDataset(CSV_FILENAME, WINDOW_SIZE, STRIDE)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    model = TinyGestureNet(num_classes=3)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    print(f"Training on {len(dataset)} windows...")
    print(f"Total Model Parameters: {sum(p.numel() for p in model.parameters())}")

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in dataloader:
            optimizer.zero_grad()

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(dataloader)
        epoch_acc = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")

    # Save the PyTorch model
    torch.save(model.state_dict(), MODEL_SAVE_PATH)
    print(f"Model saved to {MODEL_SAVE_PATH}")

    # Save the scaler mean/variance for inference in C++ / Edge device later
    np.save("scaler_mean.npy", dataset.scaler.mean_)
    np.save("scaler_scale.npy", dataset.scaler.scale_)
    print("Saved scaler parameters (required for live inference normalization).")

In [ ]:
train_model()

Training on 890 windows...
Total Model Parameters: 3683
Epoch [1/20] | Loss: 0.9270 | Accuracy: 54.94%
Epoch [2/20] | Loss: 0.7253 | Accuracy: 72.25%
Epoch [3/20] | Loss: 0.6029 | Accuracy: 73.37%
Epoch [4/20] | Loss: 0.5146 | Accuracy: 73.82%
Epoch [5/20] | Loss: 0.4755 | Accuracy: 77.75%
Epoch [6/20] | Loss: 0.4447 | Accuracy: 79.21%
Epoch [7/20] | Loss: 0.4157 | Accuracy: 80.90%
Epoch [8/20] | Loss: 0.4018 | Accuracy: 82.70%
Epoch [9/20] | Loss: 0.3944 | Accuracy: 81.24%
Epoch [10/20] | Loss: 0.3663 | Accuracy: 84.38%
Epoch [11/20] | Loss: 0.3465 | Accuracy: 84.38%
Epoch [12/20] | Loss: 0.3617 | Accuracy: 82.36%
Epoch [13/20] | Loss: 0.3492 | Accuracy: 83.93%
Epoch [14/20] | Loss: 0.3538 | Accuracy: 83.93%
Epoch [15/20] | Loss: 0.3328 | Accuracy: 84.83%
Epoch [16/20] | Loss: 0.3372 | Accuracy: 84.94%
Epoch [17/20] | Loss: 0.3198 | Accuracy: 85.73%
Epoch [18/20] | Loss: 0.3231 | Accuracy: 85.96%
Epoch [19/20] | Loss: 0.3194 | Accuracy: 85.28%
Epoch [20/20] | Loss: 0.3064 | Accuracy: 

In [ ]:
train_model(50)

Training on 890 windows...
Total Model Parameters: 3683
Epoch [1/50] | Loss: 0.9811 | Accuracy: 54.83%
Epoch [2/50] | Loss: 0.7361 | Accuracy: 72.13%
Epoch [3/50] | Loss: 0.6061 | Accuracy: 73.15%
Epoch [4/50] | Loss: 0.5365 | Accuracy: 75.06%
Epoch [5/50] | Loss: 0.4743 | Accuracy: 77.75%
Epoch [6/50] | Loss: 0.4429 | Accuracy: 79.66%
Epoch [7/50] | Loss: 0.4128 | Accuracy: 81.91%
Epoch [8/50] | Loss: 0.4025 | Accuracy: 82.36%
Epoch [9/50] | Loss: 0.3725 | Accuracy: 82.81%
Epoch [10/50] | Loss: 0.3719 | Accuracy: 85.17%
Epoch [11/50] | Loss: 0.3535 | Accuracy: 83.60%
Epoch [12/50] | Loss: 0.3386 | Accuracy: 84.72%
Epoch [13/50] | Loss: 0.3234 | Accuracy: 86.29%
Epoch [14/50] | Loss: 0.3114 | Accuracy: 86.40%
Epoch [15/50] | Loss: 0.3099 | Accuracy: 85.96%
Epoch [16/50] | Loss: 0.3003 | Accuracy: 85.62%
Epoch [17/50] | Loss: 0.2917 | Accuracy: 89.10%
Epoch [18/50] | Loss: 0.2887 | Accuracy: 86.85%
Epoch [19/50] | Loss: 0.2837 | Accuracy: 87.30%
Epoch [20/50] | Loss: 0.2786 | Accuracy: 

In [ ]:
train_model(100)

Training on 890 windows...
Total Model Parameters: 3683
Epoch [1/100] | Loss: 0.9200 | Accuracy: 55.73%
Epoch [2/100] | Loss: 0.6998 | Accuracy: 73.37%
Epoch [3/100] | Loss: 0.5848 | Accuracy: 74.61%
Epoch [4/100] | Loss: 0.5296 | Accuracy: 74.38%
Epoch [5/100] | Loss: 0.4856 | Accuracy: 75.39%
Epoch [6/100] | Loss: 0.4448 | Accuracy: 79.89%
Epoch [7/100] | Loss: 0.4240 | Accuracy: 80.22%
Epoch [8/100] | Loss: 0.4137 | Accuracy: 81.01%
Epoch [9/100] | Loss: 0.3947 | Accuracy: 82.25%
Epoch [10/100] | Loss: 0.3764 | Accuracy: 83.03%
Epoch [11/100] | Loss: 0.3645 | Accuracy: 83.93%
Epoch [12/100] | Loss: 0.3540 | Accuracy: 84.72%
Epoch [13/100] | Loss: 0.3584 | Accuracy: 83.15%
Epoch [14/100] | Loss: 0.3377 | Accuracy: 84.49%
Epoch [15/100] | Loss: 0.3334 | Accuracy: 84.83%
Epoch [16/100] | Loss: 0.3303 | Accuracy: 85.39%
Epoch [17/100] | Loss: 0.3214 | Accuracy: 85.73%
Epoch [18/100] | Loss: 0.3081 | Accuracy: 85.84%
Epoch [19/100] | Loss: 0.3039 | Accuracy: 87.64%
Epoch [20/100] | Loss: